# Stigmergic swarm - bundle path (Colab)

Loads a task-conditional **model bundle** - multiple vLLM engines resident in VRAM at once - and routes each agent role (scout, forager, critic, hater, validator, synthesizer) to the right engine.

**Hardware requirement:** H100 80 GB or A100 80 GB. The debate / coding bundles need ~75 GB total. On smaller GPUs the engines won't fit and the load will fail.

**Flow:** cells 1-5 are one-time setup (mount, paths, clone, install, Cohere). Cell 6 is the debate task. Cell 7 is the coding task. Cell 8 inspects the most recent output.

The bundle path runs `run_swarm.py` as a **shell subprocess** (`!python ...`), not in-kernel - vLLM v1's subprocess engine-core can't handshake inside the Jupyter kernel.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Configure paths

Run outputs, knowledge base, retrieval cache, Cohere FAISS index -> Drive (persistent). HuggingFace cache -> `/content/hf_cache` (ephemeral, refilled per session).

In [ ]:
import os

DRIVE_BASE = '/content/drive/MyDrive/swarm'
os.environ['SWARM_OUTPUTS_BASE_DIR']    = f'{DRIVE_BASE}/runs'
os.environ['SWARM_KB_DIR']              = f'{DRIVE_BASE}/knowledge_base'
os.environ['SWARM_RETRIEVAL_CACHE_DIR'] = f'{DRIVE_BASE}/retrieval_cache'
os.environ['SWARM_CORPORA_DIR']         = f'{DRIVE_BASE}/corpora'

for d in ('runs', 'knowledge_base', 'retrieval_cache', 'corpora'):
    os.makedirs(f'{DRIVE_BASE}/{d}', exist_ok=True)

# Ephemeral HF cache + force Colab tier-aware code paths in config.py.
os.environ['HF_HOME']             = '/content/hf_cache'
os.environ['HF_HUB_DISABLE_XET']  = '1'
os.environ['COLAB']               = '1'
os.makedirs('/content/hf_cache', exist_ok=True)

print('Persistent dirs (Drive):')
for k in ('SWARM_OUTPUTS_BASE_DIR', 'SWARM_KB_DIR', 'SWARM_RETRIEVAL_CACHE_DIR', 'SWARM_CORPORA_DIR'):
    print(f'  {k} = {os.environ[k]}')
print(f"HF_HOME = {os.environ['HF_HOME']}")

## 3. Clone (or pull) the repository

Re-running this cell on a later session does `git pull` instead of a fresh clone.

In [ ]:
import os, subprocess

REPO_URL = 'https://github.com/sfuqua6/Stigmeric-Coordination.git'
REPO_DIR = '/content/swarm_repo'

if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
else:
    print(f'{REPO_DIR} exists; running git pull')
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)

# Show what's on disk so you can confirm the bundle code is present.
out = subprocess.run(['git', '-C', REPO_DIR, 'log', '--oneline', '-3'],
                     capture_output=True, text=True)
print(out.stdout)

os.chdir(REPO_DIR)
print('cwd =', os.getcwd())

## 4. Install dependencies

`vllm` is the heavy one (~5 min, brings its own CUDA-matched torch).

In [ ]:
!pip install -q -r requirements-colab.txt
!python -c "import vllm; print(f'vllm {vllm.__version__} OK')"
!nvidia-smi --query-gpu=name,memory.free,memory.total --format=csv

## 5. Cohere setup (retrieval credentials)

Two credentials, for different reasons:

| What | Why |
|---|---|
| `COHERE_API_KEY` | Embed your query at runtime (one call per pipeline run) |
| `HF_TOKEN` | Download the ~1 GB Cohere/Wikipedia dataset from HF Hub |

Free Cohere key: https://cohere.com/. Free HF token: https://huggingface.co/settings/tokens (read-only is fine).

In [ ]:
import os

# === EDIT THESE ===
COHERE_API_KEY = ''
HF_TOKEN       = ''
# ==================

if COHERE_API_KEY:
    os.environ['COHERE_API_KEY'] = COHERE_API_KEY
if HF_TOKEN:
    os.environ['HF_TOKEN'] = HF_TOKEN
    os.environ['HUGGING_FACE_HUB_TOKEN'] = HF_TOKEN

print('COHERE_API_KEY set:', bool(os.environ.get('COHERE_API_KEY')))
print('HF_TOKEN set:      ', bool(os.environ.get('HF_TOKEN')))

## 6. Debate task (bundle = debate_analysis)

Engines loaded into VRAM at once:

| Engine | Model | Role serves |
|---|---|---|
| primary | Qwen/Qwen2.5-14B-Instruct (fp16) | scout, forager, synthesizer |
| reasoner | casperhansen/QwQ-32B-Preview-AWQ | critic, hater |
| fast | Qwen/Qwen2.5-7B-Instruct (fp16) | validator |

First run downloads ~58 GB of weights to `/content/hf_cache/` (~10 min). Subsequent runs load from cache.

In [ ]:
!cd /content/swarm_repo && python run_swarm.py debate \
    "Cities should ban private cars to fight climate change." \
    --bundle=debate_analysis --workers=24

## 7. Coding task (bundle = coding)

Engines:

| Engine | Model | Role serves |
|---|---|---|
| primary | Qwen/Qwen2.5-Coder-32B-Instruct (fp8) | scout, forager, critic, synthesizer |
| secondary | deepseek-ai/DeepSeek-Coder-V2-Lite-Instruct | hater (different family for adversarial) |
| fast | Qwen/Qwen2.5-Coder-7B-Instruct | validator |

Run **after** cell 6 finishes. The subprocess exits and frees VRAM, so the two bundles never compete for the GPU.

In [ ]:
!cd /content/swarm_repo && python run_swarm.py coding \
    "Implement a Python function that returns the longest palindromic substring of a given string in O(n^2) time." \
    --bundle=coding --workers=24

## 8. Inspect the most recent run

Every run drops `answer.txt`, `signals.json`, `summary.json`, `round_log.json`, `citations.json`, `lineage.dot`, `run_meta.json` into a timestamped subdir.

Bundle runs additionally have `bundle`, `engines`, `speculative_enabled`, `prefix_caching_enabled` in `summary.json`.

In [ ]:
import json, os
from pathlib import Path

outputs_root = Path(os.environ['SWARM_OUTPUTS_BASE_DIR']) / 'outputs'
runs = sorted(outputs_root.glob('*'), key=lambda p: p.stat().st_mtime) if outputs_root.exists() else []

if runs:
    latest = runs[-1]
    print('latest run:', latest)
    print()

    summary_path = latest / 'summary.json'
    if summary_path.exists():
        print('=== summary.json ===')
        print(json.dumps(json.loads(summary_path.read_text()), indent=2))
        print()

    answer_path = latest / 'answer.txt'
    if answer_path.exists():
        print('=== answer.txt ===')
        print(answer_path.read_text())
else:
    print(f'no runs found in {outputs_root}')

---

## Troubleshooting

- **`Engine core initialization failed. Failed core proc(s): {}`** - you tried to run the bundle in-kernel. Use the `!python run_swarm.py ...` form (cells 6 and 7), not `await run_continuous_pipeline(...)` directly.
- **OOM during weight load** - your GPU isn't 80 GB. Drop to the homogeneous path: remove the `--bundle=...` flag from cell 6 / 7 and the run will use a single ~7B model instead.
- **`SpeculativeConfig` validation error** - your `SWARM_SPECULATIVE_DRAFT` env var points to a model with a mismatched tokenizer vocab. Unset it; speculative decoding is off by default.
- **`ImportError: cannot import name 'make_bundle_router'`** - the cloned repo is stale. Re-run cell 3 to `git pull`.
- **Notebook out of date** - it's a copy that doesn't auto-sync. `File -> Open notebook -> GitHub` to re-open fresh. The cloned code under `/content/swarm_repo/` does pull via cell 3.